**ATTENTION AND TRANSFORMERS**

**TRANSFORMER ARCHITECTURE FROM NUMPY**

In [6]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
x = np.random.randn(5,4)
x

array([[-1.34989037,  2.10446391, -1.11694455,  0.1737254 ],
       [-0.02584882, -0.82679957, -0.96953877,  0.9980625 ],
       [-2.20183996, -0.61271295, -0.73490504,  0.01851574],
       [-0.2877286 , -1.54918862, -1.78806008,  0.19915076],
       [ 1.34773536,  2.10685171, -0.54971807,  0.67895467]])

In [8]:
wq = np.random.rand(4,4)
wk = np.random.rand(4,4)
wv = np.random.rand(4,4)

In [9]:
Q = np.dot(x,wq)
K = np.dot(x,wk)
V = np.dot(x,wv)

In [10]:
raw_score = (Q @ K.T)/np.sqrt(4)

In [11]:
raw_score.shape

(5, 5)

In [12]:
softmax = np.exp(raw_score) / np.sum(np.exp(raw_score),axis=1,keepdims=True)

In [13]:
output = np.dot(softmax,V)
output.shape

(5, 4)

In [14]:
print(np.sum(softmax, axis=1))

[1. 1. 1. 1. 1.]


**ATTENTION AND TRANSFORMER APPLIED TO JPMC AND S&P500 DATA**

**PREPARING THE DATA**

In [15]:
jpmc = yf.download(['JPM'],start = '2010-01-1',end='2025-01-01')
sp500 = yf.download(['^GSPC'],start = '2010-01-1',end='2025-01-01')

/tmp/ipykernel_2296/4128732890.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  jpmc = yf.download(['JPM'],start = '2010-01-1',end='2025-01-01')
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_2296/4128732890.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500 = yf.download(['^GSPC'],start = '2010-01-1',end='2025-01-01')
[*********************100%***********************]  1 of 1 completed


**EXPLORATORY DATA ANALYSIS**

In [16]:
jpmc.isnull().sum().sum()

np.int64(0)

In [17]:
sp500.isnull().sum().sum()

np.int64(0)

In [18]:
jpmc.duplicated().sum()

np.int64(0)

In [19]:
sp500.duplicated().sum()

np.int64(0)

In [20]:
jpmc.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3774 entries, 2010-01-04 to 2024-12-31
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   (Close, JPM)   3774 non-null   float64
 1   (High, JPM)    3774 non-null   float64
 2   (Low, JPM)     3774 non-null   float64
 3   (Open, JPM)    3774 non-null   float64
 4   (Volume, JPM)  3774 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 176.9 KB


In [21]:
sp500.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3774 entries, 2010-01-04 to 2024-12-31
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   (Close, ^GSPC)   3774 non-null   float64
 1   (High, ^GSPC)    3774 non-null   float64
 2   (Low, ^GSPC)     3774 non-null   float64
 3   (Open, ^GSPC)    3774 non-null   float64
 4   (Volume, ^GSPC)  3774 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 176.9 KB


**FEATURE ENGINEERING**

In [22]:
#Daily returns
jpmc['returns'] = jpmc['Close']['JPM'].pct_change(fill_method=None)

In [23]:
#volume ratio
volume = jpmc['Volume']['JPM']
jpmc['volume ratio'] = (volume/ volume.rolling(window=20).mean()).shift(1)

In [24]:
#20-day rolling return
jpmc['20-day rolling return'] = jpmc['returns'].rolling(window=20).mean().shift(1)

In [25]:
delta = jpmc['Close']['JPM'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
RSI = 100 - (100 / (1 + gain/loss))
jpmc['RSI'] = RSI.shift(1)

In [26]:
sp500['SP500 returns'] = sp500['Close']['^GSPC'].pct_change(fill_method=None)

In [27]:
sp500_return_dataframe = pd.DataFrame({
    'Date': sp500.index,
    'SP500 returns': sp500['SP500 returns']
})
sp500_return_dataframe.set_index('Date', inplace=True)

In [28]:
jpmc_clean = pd.DataFrame({
    "returns": jpmc['returns'],
    "Volume Ratio": jpmc['volume ratio'],
    "Rolling Returns": jpmc['20-day rolling return'],
    "RSI": jpmc['RSI']
})

In [29]:
jpmc_clean = jpmc_clean.join(sp500_return_dataframe,how='inner')

In [30]:
jpmc_clean['target'] = (jpmc_clean['returns'] > 0).astype(int).shift(-1)

In [31]:
jpmc_clean.dropna(inplace=True)
jpmc_clean

,returns,Volume Ratio,Rolling Returns,RSI,SP500 returns,target
Date,,,,,,
2010-02-03,-0.006412,0.853654,-0.002507,36.611987,-0.005474,0.0
2010-02-04,-0.048151,0.696517,-0.003796,31.106774,-0.031141,0.0
2010-02-05,-0.001304,1.037733,-0.006478,23.539134,0.002897,0.0
2010-02-08,-0.015666,1.327887,-0.007534,25.589729,-0.008863,1.0
2010-02-09,0.018302,1.007097,-0.008194,25.133604,0.013040,1.0
...,...,...,...,...,...,...
2024-12-23,0.003325,3.523394,-0.001417,36.638937,0.007287,1.0
2024-12-24,0.016444,0.934839,-0.002024,39.867649,0.011043,1.0
2024-12-26,0.003425,0.419782,-0.001552,48.407853,-0.000406,0.0


In [32]:
jpmc_clean.shape

(3752, 6)

In [33]:
jpmc_clean.columns

Index(['returns', 'Volume Ratio', 'Rolling Returns', 'RSI', 'SP500 returns',
       'target'],
      dtype='object')

**SEQUENCES AND SPLITTING DATA**

In [36]:
features = jpmc_clean.drop('target',axis=1)
target = jpmc_clean['target']

In [37]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

In [38]:
target = target.to_numpy()

In [40]:
lookback = 20
x,y = [],[]

for i in range (len(features_scaled) - lookback):
  x.append(features_scaled[i:i+lookback])
  y.append(target[i+lookback])

In [41]:
x = np.array(x)
y = np.array(y)

In [42]:
x.shape

(3732, 20, 5)

In [43]:
y.shape

(3732,)

In [44]:
x_train = x[:int(0.8*len(x))]
y_train = y[:int(0.8*len(y))]
x_test = x[int(0.8*len(x)):]
y_test = y[int(0.8*len(y)):]

In [45]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(2985, 20, 5)
(2985,)
(747, 20, 5)
(747,)
